**ENTRENAMIENTO BASADO EN AUTOCARAS (PCA SOBRE PÍXELES + SVM)**

En este cuaderno se desarrolla un enfoque clásico basado en la técnica de **autocaras**, proyectando la intensidad directa de los píxeles en un subespacio de baja dimensión mediante **PCA** y clasificando las proyecciones mediante una **SVM RBF**.

Este método modela la variabilidad global en intensidad. Para reducir el coste se emplean rostros a menor escala, proyectando a 150 componentes principales con blanqueamiento.

**IMPORTS NECESARIOS Y DEPENDENCIAS**

Se importan las herramientas de descomposición en componentes principales, empaquetado de pipelines y clasificación de scikit-learn, junto con las constantes y funciones auxiliares.

In [1]:
import time
import joblib
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.svm import SVC
from comun import TAM_PIX, Extractor, cargar_dataset, informe, registrar_entrenamiento

**CONFIGURACIÓN DE RUTAS Y ARCHIVO DE SALIDA**

Se configuran los directorios de datos y de salida.

In [2]:
RUTA_TRAIN = "dataset/train"
RUTA_TEST = "dataset/test"
SALIDA = "modelos/autocaras_svm.joblib"

**CARGA Y NORMALIZACIÓN DEL CONJUNTO DE DATOS**

Se cargan las imágenes redimensionándolas a una resolución adaptada de 32x32 píxeles.

In [3]:
imgs_tr, ytr = cargar_dataset(RUTA_TRAIN, tam=TAM_PIX)
imgs_te, yte = cargar_dataset(RUTA_TEST, tam=TAM_PIX)

dataset/train: 30241 imágenes | por clase: {'angry': 3995, 'duckface': 1968, 'fear': 4097, 'happy': 7215, 'neutral': 4965, 'sad': 4830, 'surprise': 3171}
dataset/test: 7396 imágenes | por clase: {'angry': 958, 'duckface': 329, 'fear': 1024, 'happy': 1774, 'neutral': 1233, 'sad': 1247, 'surprise': 831}


**EXTRACCIÓN DEL VECTOR DE INTENSIDAD DE PÍXELES**

Utilizando el método `pixeles` de `Extractor`, se transforma cada imagen bidimensional en un vector unidimensional normalizado de 1024 valores de intensidad.

In [4]:
extractor = Extractor("pixeles")
Xtr = extractor.transform(imgs_tr)
Xte = extractor.transform(imgs_te)

**DEFINICIÓN DEL PIPELINE DE AUTOCARAS**

Se diseña la arquitectura encadenando un `PCA` configurado con 150 componentes principales y blanqueamiento espectral, seguido de un clasificador `SVC` con kernel RBF y ponderación de clases balanceada.

In [5]:
clasificador = make_pipeline(
    PCA(n_components=150, whiten=True, random_state=42),
    SVC(kernel="rbf", C=5, class_weight="balanced"),
)

**ENTRENAMIENTO DEL MODELO DE AUTOCARAS**

Se ajustan simultáneamente el espacio de proyección de autocaras y el clasificador SVM con los vectores aplanados de entrenamiento, registrando el tiempo transcurrido.

In [6]:
t0 = time.time()
clasificador.fit(Xtr, ytr)
t_entreno = time.time() - t0

**EVALUACIÓN SOBRE EL CONJUNTO DE TEST**

Se proyectan las imágenes del conjunto de test sobre los autovectores aprendidos, se clasifican y se generan las estadísticas de precisión, recall y F1-score por clase.

In [7]:
t0 = time.time()
pred = clasificador.predict(Xte)
t_pred = time.time() - t0
metricas = informe(yte, pred, t_entreno, t_pred)

              precision    recall  f1-score   support

       angry       0.37      0.36      0.37       958
    duckface       0.70      0.75      0.72       329
        fear       0.40      0.36      0.38      1024
       happy       0.59      0.71      0.64      1774
     neutral       0.45      0.47      0.46      1233
         sad       0.39      0.34      0.36      1247
    surprise       0.74      0.60      0.67       831

    accuracy                           0.50      7396
   macro avg       0.52      0.51      0.52      7396
weighted avg       0.50      0.50      0.50      7396

Matriz de confusión (filas = real, columnas = predicho):
[[ 349   16  117  186  138  132   20]
 [  13  246   11   19   13   24    3]
 [ 135   13  373  149  132  157   65]
 [ 118   27   71 1251  146  134   27]
 [ 116   21   99  222  581  171   23]
 [ 160   25  180  212  208  428   34]
 [  48    3   91   69   63   55  502]]

Accuracy test: 0.5043 | F1 macro: 0.5156
Tiempo de entrenamiento: 273.7 s
Infe

**SERIALIZACIÓN Y GUARDADO DEL PIPELINE COMPLETO**

Se ensambla el extractor de píxeles junto con el clasificador entrenado en un único objeto `Pipeline` y se guarda el modelo resultante.

In [8]:
modelo = Pipeline([("extractor", extractor), ("clasificador", clasificador)])
joblib.dump(modelo, SALIDA)
print("Modelo guardado en", SALIDA)
registrar_entrenamiento(SALIDA, t_entreno, ms_imagen=1000 * t_pred / len(yte), **metricas)

Modelo guardado en modelos/autocaras_svm.joblib
Registrado en informe_entrenamiento\registro_modelos.json
